# Reto 04 — SOLUCION

No liberar hasta cerrar el modulo.

In [ ]:
import pandas as pd

RUTA = "../../../_RECURSOS/datasets/ventas.csv"

## 1 — Cargar

Los tres parametros que casi nadie pone la primera vez: `sep`, `decimal`, `encoding`.
`encoding="utf-8"` explicito evita el `UnicodeDecodeError` en Windows.

In [ ]:
ventas = pd.read_csv(RUTA, sep=";", decimal=",", encoding="utf-8")

assert ventas.shape == (491, 10)
assert ventas["precio_unitario"].dtype.kind == "f"
print("OK 1 — cargado")
ventas.head()

## 2 — Radiografia

In [ ]:
print(ventas.info())
print("\nNulos por columna:\n", ventas.isna().sum())
print("\nDuplicados exactos:", ventas.duplicated().sum())
print("\nCanales crudos:", ventas["canal"].nunique(), "->", sorted(ventas["canal"].unique()))
ventas.describe()

**Hallazgos esperados:**

1. 491 filas x 10 columnas.
2. `fecha` es `object` (texto) cuando deberia ser `datetime64`. `id_*` como texto esta bien: son codigos, no numeros con los que se opere.
3. Nulos: `ciudad` 19, `importe` 12. El resto en cero.
4. 9 filas duplicadas exactas.
5. `canal` tiene 12 valores distintos porque el mismo canal viene con espacios, en mayusculas y en minusculas. Son 3 canales reales.
6. `cantidad` tiene minimo −1: hay al menos una venta negativa.

## 3 — Duplicados

`drop_duplicates()` sin argumentos compara **todas** las columnas, asi que solo borra
filas identicas de punta a punta. Es la version segura.

Si `id_venta` se repitiera con importes distintos, no serian duplicados: serian dos hechos
distintos con el mismo codigo, y borrar uno seria perder dinero. Ahi lo correcto es
investigar el origen, no `drop_duplicates`.

In [ ]:
ventas_sin_dup = ventas.drop_duplicates().reset_index(drop=True)

assert len(ventas_sin_dup) == 482
print("OK 3 — 491 -> 482")

## 4 — Normalizar `canal`

In [ ]:
ventas_sin_dup["canal"] = ventas_sin_dup["canal"].str.strip().str.title()

assert sorted(ventas_sin_dup["canal"].unique()) == ["Telefono", "Tienda", "Web"]
print("OK 4 —", ventas_sin_dup["canal"].value_counts().to_dict())

## 5 — Fechas

`format="mixed"` deja que pandas infiera fila por fila; `dayfirst=True` resuelve la
ambiguedad de `03/07/2025` a favor de "3 de julio", que es como escribe la gente aqui.

Sin `dayfirst`, esa fecha se lee como 7 de marzo y el analisis mensual sale mal
**sin dar ningun error**. Es el bug mas silencioso de este modulo.

In [ ]:
ventas_sin_dup["fecha"] = pd.to_datetime(
    ventas_sin_dup["fecha"], format="mixed", dayfirst=True
)

assert ventas_sin_dup["fecha"].dtype.kind == "M"
assert ventas_sin_dup["fecha"].isna().sum() == 0
print("OK 5 — rango:", ventas_sin_dup["fecha"].min().date(), "a", ventas_sin_dup["fecha"].max().date())

## 6 — Nulos

`importe` es **derivado**: se recalcula, no se imputa. Imputar con la media aqui seria
inventar plata. Como la formula existe, no hay ninguna excusa para aproximar.

In [ ]:
calculado = (ventas_sin_dup["cantidad"] * ventas_sin_dup["precio_unitario"]
             * (1 - ventas_sin_dup["descuento_pct"] / 100)).round(2)

# control: donde el importe SI existe, deberia coincidir con el calculo
diff = (ventas_sin_dup["importe"] - calculado).abs()
print("Maxima diferencia donde el importe existia:", diff.max())

ventas_sin_dup["importe"] = ventas_sin_dup["importe"].fillna(calculado)

assert ventas_sin_dup["importe"].isna().sum() == 0
print("OK 6a — importe completo")

`ciudad` no es derivable. Aqui se toma la opcion de **imputar con la ciudad mas frecuente
del propio cliente**, y marcar la fila como imputada para que nadie confunda dato con estimacion.

Las otras dos opciones (borrar la fila, o `"Desconocida"`) tambien se aceptan si el alumno
las justifica. Lo que NO se acepta es rellenar sin decir nada.

In [ ]:
ventas_sin_dup["ciudad_imputada"] = ventas_sin_dup["ciudad"].isna()

moda_por_cliente = (ventas_sin_dup.dropna(subset=["ciudad"])
                    .groupby("id_cliente")["ciudad"]
                    .agg(lambda s: s.mode().iloc[0]))

ventas_sin_dup["ciudad"] = ventas_sin_dup["ciudad"].fillna(
    ventas_sin_dup["id_cliente"].map(moda_por_cliente)
).fillna("Desconocida")   # red de seguridad por si un cliente no tuviera ninguna ciudad

assert ventas_sin_dup["ciudad"].isna().sum() == 0
print("OK 6b — filas imputadas:", int(ventas_sin_dup["ciudad_imputada"].sum()))

## 7 — La cantidad negativa

In [ ]:
print(ventas_sin_dup[ventas_sin_dup["cantidad"] < 0])

`V9001`: cantidad −1, importe −3200. Es **coherente**: el importe tambien es negativo.
Eso apunta a una devolucion registrada como venta en negativo, no a un error de carga
(un error de carga tipico deja el importe positivo y solo la cantidad negativa).

Decision: **se conserva**, se marca con una columna `tipo_movimiento` y se excluye
explicitamente de los KPIs que hablan de "ventas". Borrarla cuadraria el reporte de ventas
pero descuadraria la caja.

In [ ]:
ventas_sin_dup["tipo_movimiento"] = ventas_sin_dup["cantidad"].apply(
    lambda c: "Devolucion" if c < 0 else "Venta"
)
print(ventas_sin_dup["tipo_movimiento"].value_counts().to_dict())

## 8 — Respuestas de negocio

In [ ]:
solo_ventas = ventas_sin_dup[ventas_sin_dup["tipo_movimiento"] == "Venta"]

print(f"1. Facturado 2025 : S/ {solo_ventas['importe'].sum():,.2f}")
print(f"2. Ciudad top     : {solo_ventas.groupby('ciudad')['importe'].sum().idxmax()}")
print("\n3. Ticket promedio por canal:")
print(solo_ventas.groupby("canal")["importe"].mean().round(2).to_string())
print("\n4. Facturacion por mes:")
por_mes = solo_ventas.groupby(solo_ventas["fecha"].dt.month)["importe"].sum()
print(por_mes.round(2).to_string())
print(f"\n   Mes top: {por_mes.idxmax()}")

## 9 — Exportar

`index=False` es el olvido clasico: sin el aparece una columna `Unnamed: 0` que
se arrastra a todos los pasos siguientes.

In [ ]:
ventas_sin_dup.to_csv("ventas_limpio.csv", sep=";", decimal=",",
                      index=False, encoding="utf-8")

control = pd.read_csv("ventas_limpio.csv", sep=";", decimal=",")
assert "Unnamed: 0" not in control.columns
assert control["importe"].isna().sum() == 0
print(f"OK 9 — {len(control)} filas exportadas")

---
### Errores tipicos al corregir

- `read_csv` sin `sep=";"`: sale un DataFrame de 1 columna y el alumno igual sigue.
- `read_csv` sin `decimal=","`: `precio_unitario` queda como texto y `.sum()` **concatena strings** en vez de sumar. No da error.
- `to_datetime` sin `dayfirst=True`: 124 fechas quedan con dia y mes cambiados, en silencio.
- `fillna(df["importe"].mean())`: rellenar un campo derivado con una media. Es el error conceptual mas grave del reto.
- `dropna()` a secas: se lleva 31 filas por perseguir 12 nulos.
- Limpiar `canal` con `.replace()` valor por valor: funciona con 12 variantes, no escala a 300.
- `to_csv` sin `index=False`.
- Rutas absolutas `C:\Users\...`: el notebook no corre en la maquina del profesor.